In [1]:
%pip install azure-ai-ml

Note: you may need to restart the kernel to use updated packages.


In [2]:
from azure.ai.ml import MLClient, command, dsl
from azure.ai.ml.entities import RecurrenceTrigger, RecurrencePattern, JobSchedule, Environment
from azure.identity import DefaultAzureCredential

# 1. Authenticate to your specific Workspace
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id="26b57ef9-1628-4837-a014-81f6424512c1",
    resource_group_name="rg-trading-bot-dev",
    workspace_name="ml-trading-workspace"
)

# 2. Define the Cloud Environment (The libraries needed)
ml_env = Environment(
    name="trading-cluster-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml"
)

# 3. Define the Component
clustering_component = command(
    code="./",  
    command="python market_sector_rotation_strategy.py",
    environment=ml_env,
    compute="trading-dev-machine",
    environment_variables={
        # Just pass the URL
        "KEY_VAULT_URL": "https://kv-ml-trading-workspace.vault.azure.net/"
    }
)

# 4. Wrap it in a Pipeline (Azure requires this for schedules!)
@dsl.pipeline(
    name="daily_clustering_pipeline",
    description="Single-step pipeline for market clustering and Agentic Thesis generation"
)
def my_clustering_pipeline():
    # Execute the component as step 1
    step_1 = clustering_component()

# Instantiate the pipeline
pipeline_job = my_clustering_pipeline()

# 5. Define the Timer (Runs daily at 9:30 PM)
job_schedule = JobSchedule(
    name="daily_market_clustering_schedule",
    trigger=RecurrenceTrigger(
        frequency="day",
        interval=1,
        schedule=RecurrencePattern(hours=21, minutes=30)
    ),
    create_job=pipeline_job  # <-- Notice we pass the PIPELINE now, not the command
)

# 6. Submit to Azure
ml_client.schedules.begin_create_or_update(job_schedule)
print("✅ Enterprise MLOps Schedule successfully created with secure Key Vault injections!")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

✅ Enterprise MLOps Schedule successfully created with secure Key Vault injections!
.....